In [ ]:
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import tqdm

from ase import units
from ase.atoms import Atoms
from ase.build import molecule
from torch_dftd.torch_dftd3_calculator import TorchDFTD3Calculator
from ase.calculators.dftd3 import DFTD3

from cc2cc.utils import gen_mole


class Model(nn.Module):
    """
    Fully connected neural network (dense network)
    """

    def __init__(self, device="cuda", damping="zero", **kwargs):
        super().__init__()

        # device="cuda:0" for fast GPU computation.
        self.calc = TorchDFTD3Calculator(
            device=device,
            dtype=torch.float64,
            xc="b3-lyp",
            damping=damping,
            bidirectional=False,
        )

        if damping == "zero":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 1.261),
                        kwargs.get("s18", 1.703),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": kwargs.get("rs18", 1.0),
                "alp": kwargs.get("alp", 14.0),
            }
        elif damping == "bj":
            self.param_vector = torch.nn.Parameter(
                torch.tensor(
                    [
                        kwargs.get("rs6", 0.3981),
                        kwargs.get("s18", 1.9889),
                        kwargs.get("rs18", 4.4211),
                    ],
                    dtype=torch.float64,
                    device=device,
                )
            )
            self.params = {
                "s6": kwargs.get("s6", 1.0),
                "rs6": self.param_vector[0],
                "s18": self.param_vector[1],
                "rs18": self.param_vector[2],
                "alp": kwargs.get("alp", 14.0),
            }
        self.calc.dftd_module.params = self.params
        self.damping = damping

    def forward(self, batch_dicts):
        self.calc.reset()

        # Calculate the energy using the DFTD3 calculator
        E_disp = self.calc.dftd_module.calc_energy_batch(
            **batch_dicts, damping=self.damping
        )

        return E_disp * units.mol / units.kcal

    def obtain_batch_dicts(self, atoms_list):
        # Calculator.calculate(self, atoms, properties, system_changes)
        input_dicts_list = [self.calc._preprocess_atoms(atoms) for atoms in atoms_list]
        # --- Make batch ---
        n_nodes_list = [d["Z"].shape[0] for d in input_dicts_list]
        shift_index_array = torch.cumsum(torch.tensor([0] + n_nodes_list), dim=0)
        cell_batch = torch.stack(
            [
                (
                    torch.eye(3, device=self.calc.device, dtype=self.calc.dtype)
                    if d["cell"] is None
                    else d["cell"]
                )
                for d in input_dicts_list
            ]
        )

        batch_dicts = dict(
            Z=torch.cat([d["Z"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            pos=torch.cat([d["pos"] for d in input_dicts_list], dim=0),  # (n_nodes,)
            cell=cell_batch,  # (bs, 3, 3)
            pbc=torch.stack([d["pbc"] for d in input_dicts_list]),  # (bs, 3)
            shift_pos=torch.cat(
                [d["shift_pos"] for d in input_dicts_list], dim=0
            ),  # (n_nodes,)
        )
        batch_dicts["edge_index"] = torch.cat(
            [
                d["edge_index"] + shift_index_array[i]
                for i, d in enumerate(input_dicts_list)
            ],
            dim=1,
        )
        batch_dicts["batch"] = torch.cat(
            [
                torch.full((n_nodes,), i, dtype=torch.long, device=self.calc.device)
                for i, n_nodes in enumerate(n_nodes_list)
            ],
            dim=0,
        )
        batch_dicts["batch_edge"] = torch.cat(
            [
                torch.full(
                    (d["edge_index"].shape[1],),
                    i,
                    dtype=torch.long,
                    device=self.calc.device,
                )
                for i, d in enumerate(input_dicts_list)
            ],
            dim=0,
        )

        batch_dicts["pos"].requires_grad_(True)
        return batch_dicts


data = pd.read_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-1513512_gmtkn-cc-pVDZ.csv"
)
data_name_list = (data["name"].str.split("_cc-pVDZ").str[0]).to_numpy()
data_cc_ene = data["cc_ene"].to_numpy() * 627.5094733748099
data_dft_ene = data["scf_ene"].to_numpy() * 627.5094733748099
batch_subset = [
    "W4_11",
    "G21EA",
    "G21IP",
    "DIPCS10",
    "PA26",
    "SIE4x4",
    "ALKBDE10",
    "YBDE18",
    "AL2X6",
    "HEAVYSB11",
    "NBPRC",
    "ALK8",
    "RC21",
    "G2RC",
    "BH76RC",
    "FH51",
    "TAUT15",
    "DC13",
    "MB16_43",
    "DARC",
    "RSE43",
    "BSR36",
    "CDIE20",
    "ISO34",
    # "ISOL24",
    # "C60ISO",
    "PArel",
    "BH76",
    "BHPERI",
    "BHDIV10",
    "INV24",
    "BHROT27",
    "PX13",
    "WCPT18",
    "RG18",
    "ADIM6",
    "S22",
    "S66",
    # "HEAVY28",
    "WATER27",
    "CARBHB12",
    "PNICO23",
    "HAL59",
    "AHB21",
    "CHB6",
    "IL16",
    "IDISP",
    "ICONF",
    "ACONF",
    "Amino20x4",
    "PCONF21",
    "MCONF",
    "SCONF",
    # "UPU23",
    "BUT14DIOL",
]

with open(f"new_dataset/gmtkn-cc-pVDZ.json") as f:
    json_data = json.load(f)

input_batch = {}
name_batch_list = {}
weight_batch_list = {}
mean_absolute_deviation = []
model = Model(device="cuda", damping="bj")
# model = Model(device="cuda", damping="zero")
model.compile(mode="max-autotune-no-cudagraphs")
for name_mol in data_name_list:
    for i_subset in batch_subset:
        if i_subset == "BH76RC":
            i_subset_name = "BH76"
        else:
            i_subset_name = i_subset
        if name_mol.startswith(i_subset_name):
            mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
            atoms = Atoms(
                symbols=mol.elements, positions=mol.atom_coords() * units.Bohr
            )
            if i_subset not in input_batch:
                input_batch[i_subset] = []
            input_batch[i_subset].append(atoms)
            if i_subset not in name_batch_list:
                name_batch_list[i_subset] = []
            name_batch_list[i_subset].append(name_mol)

for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]
    name_batch_list[i_subset] = np.array(name_batch_list[i_subset])
    input_batch[i_subset] = model.obtain_batch_dicts(input_batch[i_subset])
    reaction_dict_copy = reaction_dict.copy()
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict_copy.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            if col.size != 1:
                print(f"Warning: {mole_name} not found in name_list")
                reaction_dict.pop(i_reaction_keys)
                break
    json_data[f"reaction-{i_subset}"] = reaction_dict

energy_batch_target = {}
for i_subset in batch_subset:
    if i_subset == "BH76RC":
        i_subset_name = "BH76"
    else:
        i_subset_name = i_subset
    reaction_dict = json_data[f"reaction-{i_subset}"]

    energy_batch_target[i_subset] = torch.zeros(
        len(reaction_dict), dtype=torch.float64
    )
    weight_batch = np.zeros(len(reaction_dict), dtype=np.float64)
    for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
        reaction_dict.items()
    ):
        systems_list = i_reaction["systems"]
        stoichiometry_list = i_reaction["stoichiometry"]
        energy_dft = 0

        for i in range(len(systems_list)):
            if i_subset == "BH76RC":
                mole_name = f"{systems_list[i]}"
            else:
                mole_name = f"{i_subset}-{systems_list[i]}"
            stoichiometry = int(stoichiometry_list[i])

            if mole_name in json_data:
                if isinstance(json_data[mole_name], str):
                    mole_name = json_data[mole_name]

            col = np.where(data_name_list == mole_name)[0]
            energy_dft += (data_cc_ene[col[0]] - data_dft_ene[col[0]]) * stoichiometry
            weight_batch[i_reaction_name] += data_cc_ene[col[0]] * stoichiometry
        energy_batch_target[i_subset][i_reaction_name] = energy_dft
    mean_absolute_deviation.extend(np.abs(weight_batch))
    weight_batch_list[i_subset] = 1 / np.mean(np.abs(weight_batch))

print(
    f"mean_absolute_deviation: {np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}"
)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-5)
loss_function = torch.nn.L1Loss(reduction="sum")
torch.set_printoptions(precision=5, sci_mode=False)
energy_batch_output = {}
print("start training...")

def printable(epoch):
    if epoch % 100 == 0:
        return True
    return False
if_print_step = True

for epoch in tqdm.tqdm(range(2501)):
    loss_batch = []
    wtmad_2 = 0
    optimizer.zero_grad()
    for i_subset in batch_subset:
        energy = model(input_batch[i_subset])

        reaction_dict = json_data[f"reaction-{i_subset}"]
        energy_batch_output[i_subset] = torch.zeros(
            len(reaction_dict), dtype=torch.float64
        )
        for i_reaction_name, (i_reaction_keys, i_reaction) in enumerate(
            reaction_dict.items()
        ):
            systems_list = i_reaction["systems"]
            stoichiometry_list = i_reaction["stoichiometry"]
            energy_dft = 0

            for i in range(len(systems_list)):
                mole_name = (
                    systems_list[i]
                    if i_subset == "BH76RC"
                    else f"{i_subset}-{systems_list[i]}"
                )
                stoichiometry = int(stoichiometry_list[i])

                if mole_name in json_data:
                    if isinstance(json_data[mole_name], str):
                        mole_name = json_data[mole_name]

                col_disp = np.where(name_batch_list[i_subset] == mole_name)[0]
                if col_disp.size == 1:
                    energy_dft += energy[col_disp[0]] * stoichiometry
                else:
                    print(f"Warning: {mole_name} not found in name_list")
                    break
            energy_batch_output[i_subset][i_reaction_name] = energy_dft
        loss = (
            loss_function(energy_batch_output[i_subset], energy_batch_target[i_subset])
            * weight_batch_list[i_subset]
        )
        loss_batch.append(
            torch.mean(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            ).item()
        )
        if printable(epoch) and if_print_step:
            print(
                f"{i_subset}, params: {model.param_vector.data}, mean(|E|): {torch.mean(torch.abs(energy_batch_target[i_subset])).item()}, EACH: {energy_batch_output[i_subset] - energy_batch_target[i_subset]}"
            )
        wtmad_2 += (
            torch.sum(
                torch.abs(energy_batch_output[i_subset] - energy_batch_target[i_subset])
            )
            * weight_batch_list[i_subset]
        ).item()
        # clip the loss to avoid exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        loss.backward()
    optimizer.step()

    if printable(epoch):
        print(
            f"Epoch: {epoch}, wtmad_2: {wtmad_2 * np.mean(mean_absolute_deviation) / len(mean_absolute_deviation)}, loss: {loss_batch}"
        )

print(f"params_vector {model.params}")

mean_absolute_deviation: 0.051378935483546606
start training...


  0%|          | 0/2501 [00:00<?, ?it/s]

W4_11, params: tensor([0.39810, 1.98890, 4.42110], device='cuda:0', dtype=torch.float64), mean(|E|): 8.912932139655672, EACH: tensor([    -2.30013,      0.53794,      1.03246,      1.29847,      0.37621,
            -0.28355,      0.06165,      1.89525,     10.35723,      0.28638,
             0.24946,     -4.47341,      9.99628,      0.75359,      3.16423,
             6.01381,     -0.22225,      1.50342,      0.32446,      0.98313,
            10.89390,      3.95256,      7.26138,      5.91581,     -0.04297,
             8.25969,      5.68296,      1.94281,      5.14819,     -0.21476,
             0.54850,     -0.14887,      6.84193,     16.70592,     -0.03127,
             0.25215,      3.88717,      0.37047,      7.61492,      8.83306,
             8.15042,     34.83183,     23.20073,     12.19939,     11.90074,
            14.94296,     28.09633,     23.26021,      5.64365,      5.58089,
             3.94826,     11.97909,     13.22370,      3.35202,      4.41649,
             5.5

  0%|          | 1/2501 [00:27<18:45:57, 27.02s/it]

Amino20x4, params: tensor([0.39810, 1.98890, 4.42110], device='cuda:0', dtype=torch.float64), mean(|E|): 0.6688013691527885, EACH: tensor([-0.90980, -0.66906, -0.33270,  0.96615,  1.19779,  1.13219,  0.51013,
         1.05802,  0.45084,  0.59839,  0.53141,  1.75126,  0.24224, -0.37823,
        -1.56774,  0.50680,  0.30776,  0.04318,  0.28412,  0.21384, -0.37688,
         0.06848,  0.62966,  0.32691,  0.51469,  0.86399,  0.83504,  0.38380,
         0.59911,  0.42001,  0.28354, -0.13590,  0.98630,  0.63210,  1.08784,
         1.53483, -0.08785, -0.68307, -0.07336, -0.37409, -0.16103, -0.56352,
        -0.63823, -0.67014, -0.10414,  0.09145, -0.04928,  0.18823, -0.59090,
         0.58085,  0.24187,  0.67225,  0.26893, -0.51557, -0.55830, -0.19379,
         0.01565,  0.31504,  0.31106,  0.63090,  0.15031, -0.08530,  0.82526,
        -0.54214,  1.77922,  0.27815,  1.31891,  0.86789, -0.52012, -1.18015,
        -0.89132, -0.65774,  0.18421,  0.06343, -0.09302, -0.60386, -1.07157,
        -0.

...

W4_11, params: tensor([0.40807, 1.97897, 4.43106], device='cuda:0', dtype=torch.float64), mean(|E|): 8.912932139655672, EACH: tensor([    -2.30479,      0.45414,      0.99226,      1.17105,      0.32338,
            -0.30396,      0.03063,      1.86015,      9.97737,      0.23591,
             0.18113,     -4.69352,      9.85921,      0.65716,      2.97397,
             5.67425,     -0.25353,      1.48725,      0.26796,      0.95649,
            10.69094,      3.79924,      7.18454,      5.63034,     -0.08597,
             8.01598,      5.55372,      1.79727,      5.03727,     -0.23735,
             0.52181,     -0.16069,      6.71156,     16.63716,     -0.05151,
             0.24584,      3.77054,      0.36162,      7.40712,      8.63802,
             7.93487,     34.65514,     23.12131,     12.04384,     11.71749,
            14.85566,     27.98759,     23.21207,      5.54261,      5.46275,
             3.84068,     11.93400,     12.97195,      3.26399,      4.34726,
             5.4

  4%|▍         | 101/2501 [02:28<47:21,  1.18s/it]

Amino20x4, params: tensor([0.40807, 1.97897, 4.43106], device='cuda:0', dtype=torch.float64), mean(|E|): 0.6688013691527885, EACH: tensor([-0.87437, -0.62855, -0.30212,  0.98076,  1.17150,  1.13774,  0.50440,
         1.10016,  0.44138,  0.57033,  0.49327,  1.70680,  0.18834, -0.39986,
        -1.55141,  0.47323,  0.31550,  0.05050,  0.28284,  0.22904, -0.34799,
         0.06250,  0.63762,  0.34202,  0.49246,  0.82103,  0.79117,  0.44252,
         0.56220,  0.39967,  0.26578, -0.12232,  0.95992,  0.59730,  1.07052,
         1.51964, -0.11320, -0.70136, -0.09559, -0.39819, -0.16553, -0.53618,
        -0.63841, -0.63942, -0.10190,  0.09112, -0.03231,  0.22001, -0.63241,
         0.49337,  0.20264,  0.60101,  0.23427, -0.49722, -0.56949, -0.17713,
        -0.00457,  0.31463,  0.30653,  0.62401,  0.13952, -0.09739,  0.79442,
        -0.56894,  1.74615,  0.28691,  1.31786,  0.87217, -0.53892, -1.10367,
        -0.82286, -0.61965,  0.18640,  0.03240, -0.10169, -0.58294, -1.06634,
        -0.

  6%|▋         | 157/2501 [03:33<53:14,  1.36s/it]


KeyboardInterrupt: 

In [6]:
data_dft_bj = []

for name_mol in data_name_list:
    mol = gen_mole(name_mol, 0, 1, 0, "cc-pVDZ", True, "gmtkn-cc-pVDZ")
    atoms = Atoms(symbols=mol.elements, positions=mol.atom_coords() * units.Bohr)
    energy = model(model.obtain_batch_dicts([atoms]))
    print(f"{name_mol}: {energy.item():.10f} kcal/mol")
    data_dft_bj.append(energy.item() / 627.5094733748099)

data["modified_dft_d3zero"] = data_dft_bj
data.to_csv(
    "/home/dhem/workspace/2025.1/validate/ccdft_cc-pVDZ_atom-1-2453600_gmtkn-cc-pVDZ.csv",
    index=False,
)

BSR36-ch4: -0.2174092661 kcal/mol
BSR36-c2h6: -2.3199845812 kcal/mol
BSR36-r1: -10.6162756838 kcal/mol
BSR36-h1: -18.3407325456 kcal/mol
BSR36-c4: -53.0955226743 kcal/mol


# modified_ai_d3zero
params_vector {'s6': 1.0, 'rs6': tensor(1.5055195208, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 's18': tensor(1.4496851945, device='cuda:0', dtype=torch.float64,
       grad_fn=<AsStridedBackward0>), 'rs18': 1.0, 'alp': 14.0}